<a href="https://colab.research.google.com/github/JoelDrake302/ML-AutoTheft/blob/Faheem_branch/Auto_theft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import requests
import re

# DATA PREPARATION

load the original file from Toronto Police

In [2]:
file_id = '190IgZ2OhktJOw7K7Fws_yMkWzQVznmOp'
download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

# Download the file
response = requests.get(download_url)
response.raise_for_status() # Raise an exception for bad status codes

# Save the file locally
local_file_path = '/content/Auto_Theft_Open_Data.csv'
with open(local_file_path, 'wb') as f:
    f.write(response.content)

df = pd.read_csv(local_file_path, encoding="utf-8")

In [3]:
df.head()

,OBJECTID,EVENT_UNIQUE_ID,REPORT_DATE,OCC_DATE,REPORT_YEAR,REPORT_MONTH,REPORT_DAY,REPORT_DOY,REPORT_DOW,REPORT_HOUR,...,CSI_CATEGORY,HOOD_158,NEIGHBOURHOOD_158,Region,HOOD_140,NEIGHBOURHOOD_140,LONG_WGS84,LAT_WGS84,x,y
0,1,GO-20141262837,01/01/14 5:00,12/25/2013 5:00:00 AM,2014,January,1,1,Wednesday,15,...,Auto Theft,159,Etobicoke City Centre (159),Region. 1,14,Islington-City Centre West (14),-79.529692,43.618988,-8853204.784,5406667.722
1,2,GO-20141263217,01/01/14 5:00,12/31/2013 5:00:00 AM,2014,January,1,1,Wednesday,16,...,Auto Theft,43,Victoria Village (43),NaN,43,Victoria Village (43),-79.306754,43.734654,-8828387.423,5424470.688
2,3,GO-20141262914,01/01/14 5:00,01/01/14 5:00,2014,January,1,1,Wednesday,15,...,Auto Theft,123,Cliffcrest (123),NaN,123,Cliffcrest (123),-79.236119,43.721827,-8820524.401,5422494.748
3,4,GO-20141266240,01/02/14 5:00,01/02/14 5:00,2014,January,2,2,Thursday,9,...,Auto Theft,60,Woodbine-Lumsden (60),NaN,60,Woodbine-Lumsden (60),-79.313796,43.688101,-8829171.433,5417301.224
4,5,GO-20141266097,01/02/14 5:00,01/02/14 5:00,2014,January,2,2,Thursday,8,...,Auto Theft,129,Agincourt North (129),NaN,129,Agincourt North (129),-79.273925,43.813557,-8824733.012,5436634.962


# Dropping irrelevant columns

In [4]:
df.columns

Index(['OBJECTID', 'EVENT_UNIQUE_ID', 'REPORT_DATE', 'OCC_DATE', 'REPORT_YEAR',
       'REPORT_MONTH', 'REPORT_DAY', 'REPORT_DOY', 'REPORT_DOW', 'REPORT_HOUR',
       'OCC_YEAR', 'OCC_MONTH', 'OCC_DAY', 'OCC_DOY', 'OCC_DOW', 'OCC_HOUR',
       'DIVISION', 'LOCATION_TYPE', 'PREMISES_TYPE', 'UCR_CODE', 'UCR_EXT',
       'OFFENCE', 'CSI_CATEGORY', 'HOOD_158', 'NEIGHBOURHOOD_158', 'Region',
       'HOOD_140', 'NEIGHBOURHOOD_140', 'LONG_WGS84', 'LAT_WGS84', 'x', 'y'],
      dtype='object')

In [5]:
columns_to_drop = ['OBJECTID', 'EVENT_UNIQUE_ID', 'REPORT_DATE', 'OCC_DATE', 'REPORT_YEAR','REPORT_MONTH', 'REPORT_DAY', 'REPORT_DOY', 'REPORT_DOW', 'REPORT_HOUR','UCR_CODE', 'UCR_EXT','OFFENCE','CSI_CATEGORY','HOOD_140', 'NEIGHBOURHOOD_140', 'Region', 'x', 'y', 'DIVISION']

# Replace with your column names
new_df = df.drop(columns=columns_to_drop, errors='ignore')

print("Columns after dropping:", new_df.columns)

Columns after dropping: Index(['OCC_YEAR', 'OCC_MONTH', 'OCC_DAY', 'OCC_DOY', 'OCC_DOW', 'OCC_HOUR',
       'LOCATION_TYPE', 'PREMISES_TYPE', 'HOOD_158', 'NEIGHBOURHOOD_158',
       'LONG_WGS84', 'LAT_WGS84'],
      dtype='object')


In [6]:
new_df.head()

,OCC_YEAR,OCC_MONTH,OCC_DAY,OCC_DOY,OCC_DOW,OCC_HOUR,LOCATION_TYPE,PREMISES_TYPE,HOOD_158,NEIGHBOURHOOD_158,LONG_WGS84,LAT_WGS84
0,2013.0,December,25.0,359.0,Wednesday,0,"Parking Lots (Apt., Commercial Or Non-Commercial)",Outside,159,Etobicoke City Centre (159),-79.529692,43.618988
1,2013.0,December,31.0,365.0,Tuesday,17,"Apartment (Rooming House, Condo)",Apartment,43,Victoria Village (43),-79.306754,43.734654
2,2014.0,January,1.0,1.0,Wednesday,15,"Streets, Roads, Highways (Bicycle Path, Privat...",Outside,123,Cliffcrest (123),-79.236119,43.721827
3,2014.0,January,2.0,2.0,Thursday,9,"Streets, Roads, Highways (Bicycle Path, Privat...",Outside,60,Woodbine-Lumsden (60),-79.313796,43.688101
4,2014.0,January,2.0,2.0,Thursday,1,"Single Home, House (Attach Garage, Cottage, Mo...",House,129,Agincourt North (129),-79.273925,43.813557


In [7]:
grouped_df = new_df.groupby(['LOCATION_TYPE', 'PREMISES_TYPE', 'NEIGHBOURHOOD_158']).size().reset_index(name='COUNT')
print(grouped_df.head())

                      LOCATION_TYPE PREMISES_TYPE  \
0  Apartment (Rooming House, Condo)     Apartment   
1  Apartment (Rooming House, Condo)     Apartment   
2  Apartment (Rooming House, Condo)     Apartment   
3  Apartment (Rooming House, Condo)     Apartment   
4  Apartment (Rooming House, Condo)     Apartment   

                    NEIGHBOURHOOD_158  COUNT  
0               Agincourt North (129)     15  
1  Agincourt South-Malvern West (128)     17  
2                      Alderwood (20)      5  
3                          Annex (95)     14  
4                      Avondale (153)     28  


In [8]:
grouped_df.describe()

,COUNT
count,1802.000000
mean,42.591010
std,117.181436
min,1.000000
25%,1.000000
50%,5.000000
75%,33.750000
max,3237.000000


In [9]:
grouped_df['COUNT'].value_counts()

,count
COUNT,
1,469
2,207
3,99
4,88
5,59
...,...
175,1
126,1
143,1


In [10]:
Q1 = grouped_df['COUNT'].quantile(0.25)
Q3 = grouped_df['COUNT'].quantile(0.75)

def assign_risk_level(count):
    if count <= Q1:
        return 'Low Risk'
    elif count > Q1 and count <= Q3:
        return 'Medium Risk'
    else:
        return 'High Risk'

grouped_df['RISK_LEVEL'] = grouped_df['COUNT'].apply(assign_risk_level)
print(grouped_df.head())

                      LOCATION_TYPE PREMISES_TYPE  \
0  Apartment (Rooming House, Condo)     Apartment   
1  Apartment (Rooming House, Condo)     Apartment   
2  Apartment (Rooming House, Condo)     Apartment   
3  Apartment (Rooming House, Condo)     Apartment   
4  Apartment (Rooming House, Condo)     Apartment   

                    NEIGHBOURHOOD_158  COUNT   RISK_LEVEL  
0               Agincourt North (129)     15  Medium Risk  
1  Agincourt South-Malvern West (128)     17  Medium Risk  
2                      Alderwood (20)      5  Medium Risk  
3                          Annex (95)     14  Medium Risk  
4                      Avondale (153)     28  Medium Risk  


In [11]:
grouped_df['RISK_LEVEL'].value_counts()

,count
RISK_LEVEL,
Medium Risk,882
Low Risk,469
High Risk,451


## Training the Model using a 25 to 75 split

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier # Changed from LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

# Prepare features (X) and target (y) for classification
X = grouped_df[['LOCATION_TYPE', 'PREMISES_TYPE', 'NEIGHBOURHOOD_158']].copy()
y = grouped_df['RISK_LEVEL'].copy()

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, columns=['LOCATION_TYPE', 'PREMISES_TYPE', 'NEIGHBOURHOOD_158'], drop_first=True)

# Encode the target variable (RISK_LEVEL) as it's categorical
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Perform train-test split (25% train, 75% test)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.75, random_state=42)

# Initialize and train a K-Nearest Neighbors classifier (using n_neighbors=5 as a default)
classifier = KNeighborsClassifier(n_neighbors=5) # Changed from LogisticRegression
classifier.fit(X_train, y_train)

# Make predictions on the test set
y_pred = classifier.predict(X_test)

# Evaluate the model's accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Classification Model Accuracy: {accuracy:.4f}")
print(f"Classes: {le.classes_}")

Classification Model Accuracy: 0.6657
Classes: ['High Risk' 'Low Risk' 'Medium Risk']


In [ ]:
import folium


# Drop rows where latitude or longitude are missing
df.dropna(subset=['LAT_WGS84', 'LONG_WGS84'], inplace=True)

# Calculate the mean latitude and longitude to center the map
mean_lat = df['LAT_WGS84'].mean()
mean_lon = df['LONG_WGS84'].mean()

# Create a base map centered at the average coordinates
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=11)

# Add markers for a sample of the theft locations to avoid overcrowding the map
# For a very large dataset, consider a HeatMap or MarkerCluster
sample_df = df.sample(n=min(len(df), 1000), random_state=42) # Sample up to 1000 points

for idx, row in sample_df.iterrows():
    folium.Marker(
        location=[row['LAT_WGS84'], row['LONG_WGS84']],
        popup=f"OBJECTID: {row['OBJECTID']}<br>Report Date: {row['REPORT_DATE']}"
    ).add_to(m)

# Display the map
m

make the column names alphanumeric and replace space with underscore

In [ ]:
df['HOOD_158'] = pd.to_numeric(df['HOOD_158'], errors='coerce')
df = df.dropna(subset=['HOOD_158'])     # remove rows where conversion failed
df['HOOD_158'] = df['HOOD_158'].astype(int)

group by hood_58, report_year, report_month to create the count column containing the total thefts for that period

In [ ]:
# function to assign zone
def assign_zone(hood):
    if hood <= 31:
        return "ZONE_1"
    elif hood <= 63:
        return "ZONE_2"
    elif hood <= 95:
        return "ZONE_3"
    elif hood <= 127:
        return "ZONE_4"
    else:
        return "ZONE_5"

# add zone column
grouped_df['ZONE'] = grouped_df['HOOD_158'].apply(assign_zone)

df = grouped_df
print(df.head())

   HOOD_158  REPORT_YEAR REPORT_MONTH  TOTAL    ZONE
0         1         2014        April     28  ZONE_1
1         1         2014       August     45  ZONE_1
2         1         2014     December     27  ZONE_1
3         1         2014     February     19  ZONE_1
4         1         2014      January     19  ZONE_1


Adding PREV_MONTH_COUNT and NEXT_MONTH_COUNT as additional data points

*The accuracy is kind lower without these columns

In [ ]:

# clean month text
df["REPORT_MONTH"] = df["REPORT_MONTH"].astype(str).str.strip().str.title()

# convert month name to number
df["REPORT_MONTH"] = pd.to_datetime(
    df["REPORT_MONTH"], format="%B", errors="coerce"
).dt.month


df_main = df.copy()

# --- compute previous month ---
df_prev = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_prev['REPORT_MONTH'] = df_prev['REPORT_MONTH'] + 1
df_prev.loc[df_prev['REPORT_MONTH'] == 13, 'REPORT_MONTH'] = 1
df_prev.loc[df_prev['REPORT_MONTH'] == 1, 'REPORT_YEAR'] += 1

df_prev = df_prev.rename(columns={'TOTAL':'PREV_MONTH_COUNT'})

# merge
df_main = df_main.merge(
    df_prev,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

# --- compute next month ---
df_next = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_next['REPORT_MONTH'] = df_next['REPORT_MONTH'] - 1
df_next.loc[df_next['REPORT_MONTH'] == 0, 'REPORT_MONTH'] = 12
df_next.loc[df_next['REPORT_MONTH'] == 12, 'REPORT_YEAR'] -= 1

df_next = df_next.rename(columns={'TOTAL':'NEXT_MONTH_COUNT'})

# merge
df_main = df_main.merge(
    df_next,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

df = df_main.dropna(subset=['NEXT_MONTH_COUNT', 'PREV_MONTH_COUNT']).copy()
print(df.head())

   HOOD_158  REPORT_YEAR  REPORT_MONTH  TOTAL    ZONE  PREV_MONTH_COUNT  \
0         1         2014             4     28  ZONE_1              23.0   
1         1         2014             8     45  ZONE_1              24.0   
2         1         2014            12     27  ZONE_1              29.0   
3         1         2014             2     19  ZONE_1              19.0   
5         1         2014             7     24  ZONE_1              23.0   

   NEXT_MONTH_COUNT  
0              34.0  
1              12.0  
2              23.0  
3              23.0  
5              45.0  


In [ ]:
df.head()
df.tail()
df.describe()

,HOOD_158,REPORT_YEAR,REPORT_MONTH,TOTAL,PREV_MONTH_COUNT,NEXT_MONTH_COUNT
count,13093.000000,13093.000000,13093.000000,13093.000000,13093.000000,13093.000000
mean,84.752005,2020.319560,6.685328,4.934087,4.896357,4.936455
std,53.202179,3.288094,3.390281,5.777127,5.761546,5.796076
min,1.000000,2014.000000,1.000000,1.000000,1.000000,1.000000
25%,34.000000,2018.000000,4.000000,2.000000,2.000000,2.000000
50%,87.000000,2021.000000,7.000000,3.000000,3.000000,3.000000
75%,133.000000,2023.000000,10.000000,6.000000,6.000000,6.000000
max,174.000000,2025.000000,12.000000,106.000000,106.000000,106.000000


Adding Prev_3 and Prev_1 month data
**but the impact on accuracy is low**


In [ ]:
'''# clean month text
df["REPORT_MONTH"] = df["REPORT_MONTH"].astype(str).str.strip().str.title()

# convert month name to number
df["REPORT_MONTH"] = pd.to_datetime(
    df["REPORT_MONTH"], format="%B", errors="coerce"
).dt.month

df_main = df.copy()

# --- PREV 1 MONTH ---
df_prev1 = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_prev1['REPORT_MONTH'] = df_prev1['REPORT_MONTH'] + 1
df_prev1.loc[df_prev1['REPORT_MONTH'] == 13, 'REPORT_MONTH'] = 1
df_prev1.loc[df_prev1['REPORT_MONTH'] == 1, 'REPORT_YEAR'] += 1

df_prev1 = df_prev1.rename(columns={'TOTAL':'PREV_MONTH_COUNT'})

df_main = df_main.merge(
    df_prev1,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

# --- PREV 2 MONTHS ---
df_prev2 = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_prev2['REPORT_MONTH'] = df_prev2['REPORT_MONTH'] + 2
df_prev2.loc[df_prev2['REPORT_MONTH'] > 12, 'REPORT_YEAR'] += 1
df_prev2['REPORT_MONTH'] = ((df_prev2['REPORT_MONTH'] - 1) % 12) + 1

df_prev2 = df_prev2.rename(columns={'TOTAL':'PREV_2_MONTH_COUNT'})

df_main = df_main.merge(
    df_prev2,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

# --- PREV 3 MONTHS ---
df_prev3 = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_prev3['REPORT_MONTH'] = df_prev3['REPORT_MONTH'] + 3
df_prev3.loc[df_prev3['REPORT_MONTH'] > 12, 'REPORT_YEAR'] += 1
df_prev3['REPORT_MONTH'] = ((df_prev3['REPORT_MONTH'] - 1) % 12) + 1

df_prev3 = df_prev3.rename(columns={'TOTAL':'PREV_3_MONTH_COUNT'})

df_main = df_main.merge(
    df_prev3,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

# --- NEXT MONTH (TARGET) ---
df_next = df[['HOOD_158','REPORT_YEAR','REPORT_MONTH','TOTAL']].copy()

df_next['REPORT_MONTH'] = df_next['REPORT_MONTH'] - 1
df_next.loc[df_next['REPORT_MONTH'] == 0, 'REPORT_MONTH'] = 12
df_next.loc[df_next['REPORT_MONTH'] == 12, 'REPORT_YEAR'] -= 1

df_next = df_next.rename(columns={'TOTAL':'NEXT_MONTH_COUNT'})

df_main = df_main.merge(
    df_next,
    on=['HOOD_158','REPORT_YEAR','REPORT_MONTH'],
    how='left'
)

# remove rows where lag values are missing
df = df_main.dropna(
    subset=['NEXT_MONTH_COUNT','PREV_MONTH_COUNT','PREV_2_MONTH_COUNT','PREV_3_MONTH_COUNT']
).copy()'''

'# clean month text\ndf["REPORT_MONTH"] = df["REPORT_MONTH"].astype(str).str.strip().str.title()\n\n# convert month name to number\ndf["REPORT_MONTH"] = pd.to_datetime(\n    df["REPORT_MONTH"], format="%B", errors="coerce"\n).dt.month\n\ndf_main = df.copy()\n\n# --- PREV 1 MONTH ---\ndf_prev1 = df[[\'HOOD_158\',\'REPORT_YEAR\',\'REPORT_MONTH\',\'TOTAL\']].copy()\n\ndf_prev1[\'REPORT_MONTH\'] = df_prev1[\'REPORT_MONTH\'] + 1\ndf_prev1.loc[df_prev1[\'REPORT_MONTH\'] == 13, \'REPORT_MONTH\'] = 1\ndf_prev1.loc[df_prev1[\'REPORT_MONTH\'] == 1, \'REPORT_YEAR\'] += 1\n\ndf_prev1 = df_prev1.rename(columns={\'TOTAL\':\'PREV_MONTH_COUNT\'})\n\ndf_main = df_main.merge(\n    df_prev1,\n    on=[\'HOOD_158\',\'REPORT_YEAR\',\'REPORT_MONTH\'],\n    how=\'left\'\n)\n\n# --- PREV 2 MONTHS ---\ndf_prev2 = df[[\'HOOD_158\',\'REPORT_YEAR\',\'REPORT_MONTH\',\'TOTAL\']].copy()\n\ndf_prev2[\'REPORT_MONTH\'] = df_prev2[\'REPORT_MONTH\'] + 2\ndf_prev2.loc[df_prev2[\'REPORT_MONTH\'] > 12, \'REPORT_YEAR\'] += 1\

one-hot encoding

In [ ]:
df = pd.get_dummies(df, columns=["ZONE", "REPORT_MONTH", "HOOD_158"])
#list(df.columns)

Saving the new file to a file

In [ ]:
df.to_csv("cleaned_data.csv", index=False)

# MODELLING

loading the data

In [ ]:
df = pd.read_csv("/content/cleaned_data.csv", encoding="utf-8")

Splitting the data

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.3, random_state=42)

In [ ]:
target = 'NEXT_MONTH_COUNT'
features = list(train_set.columns)
features = [f for f in features if f!=target]

In [ ]:
X_tr = train_set[features]
y_tr = train_set[[target]]

X_te = test_set[features]
y_te = test_set[[target]]

Scaling the data

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_tr)

X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

##Regression

We would like to predict the number of thefts for the next month.

**LinearRegression**

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import numpy as np

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())

In [ ]:
from sklearn.linear_model import LinearRegression

lin_scores = cross_val_score(LinearRegression(), train_set[features], train_set[target], scoring="neg_mean_squared_error", cv=4)

lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)


Scores: [3.03446362 3.04482917 3.23695806 2.84530129]
Mean: 3.040388035180575


##Classification

We would like to predict if next month is high risk or low risk for for theft.

Creating another yhat which now represents if the next month is high risk or low risk for auto theft. The basis will be the median value of NEXT_MONTH_COUNT. Count that is higher than the median is considered as high risk, otherwise, it is low risk.

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
median_theft_count = np.median(df[['NEXT_MONTH_COUNT']])

In [ ]:
y_tr_b = 1*np.ravel(y_tr>=median_theft_count)
y_te_b = 1*np.ravel(y_te>=median_theft_count)

**Linear SVC**

In [ ]:
#this is predicting of the price is more than the midean using linear SVM
from sklearn.svm import LinearSVC

lin_clf = LinearSVC(random_state=42)
lin_clf.fit(X_tr, y_tr_b)

y_pred = lin_clf.predict(X_te)
accuracy_score(y_te_b, y_pred)

0.7337067209775967

**SVC**

In [ ]:
from sklearn.svm import SVC

In [ ]:
svc_clf = SVC(random_state=42, C=1.0, gamma='scale')
svc_clf.fit(X_tr, y_tr_b)
y_pred_svc = svc_clf.predict(X_te)
accuracy_score(y_te_b, y_pred_svc)

lin_clf = SVC(random_state=42)
lin_clf.fit(X_tr, y_tr_b)

y_pred = lin_clf.predict(X_tr)
accuracy_score(y_tr_b, y_pred)

0.7579923622476814

### Randomized CV

In [ ]:
'''
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

param_distributions = {
    'C': uniform(300, 500),
    'gamma': reciprocal(0.01, 0.1)
}

rnd_search = RandomizedSearchCV(
    SVC(random_state=42),
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

rnd_search.fit(X_tr, y_tr_b)

rnd_search.best_params_
rnd_search.best_score_'''